# Smart City MLOps: Nova Stad Verkeersmanagementsysteem
**Auteurs:** Nika de Vries, Thida Churam, Soner Yigitdol

Dit notebook bevat de Machine Learning pipeline voor het Nova Stad project, uitgevoerd in een lokale VS Code / GitHub omgeving. Het project bestaat uit twee componenten:
1. **Cloud Model (Tabeldata):** Verkeersvolume voorspellen m.b.v. PySpark en Random Forest.
2. **Edge Model (Beeldata):** Realtime objectdetectie op kruispunten met YOLOv8 (geëxporteerd naar TFLite).

We hanteren strikte MLOps-standaarden: alle experimenten worden gelogd via een lokale MLflow tracking server, en de PySpark-code is schaalbaar ontworpen (`local[*]`) zodat deze direct naar een cloud-cluster gemigreerd kan worden.

In [0]:
# Installeer de benodigde packages (run dit 1x in VS Code of via je requirements.txt)
%pip install pyspark ultralytics mlflow evidently pandas numpy

In [0]:
import os

# 1. Cloud Data (Tabel) Paden
TRAIN_DATA_PATH = "../data/raw/metro_interstate_traffic_volume_train.csv"
TEST_DATA_PATH  = "../data/raw/metro_interstate_traffic_volume_test.csv"

# 2. Edge Data (Beelden) Paden
# Let op: we gebruiken /*.jpg voor de PySpark inlezing
IMG_TRAIN_PATH  = "../data/bdd/images/train/*.jpg"
LABEL_TRAIN_PATH = "../data/bdd/labels/train/*.txt"

# 3. MLflow Tracking Setup
# Dit maakt een map genaamd 'mlruns' aan in je huidige werkmap
MLFLOW_TRACKING_URI = "file://" + os.path.abspath("./mlruns")

In [0]:
import time
import mlflow
import numpy as np
import pandas as pd

# PySpark Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, to_timestamp, year, month, dayofmonth, hour, dayofweek
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

# YOLO Imports
from ultralytics import YOLO

# Start een lokale Spark sessie met alle CPU cores ("local[*]")
spark = SparkSession.builder \
    .appName("NovaStad_Pipeline_Local") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Lokale Spark Sessie gestart")

## 1. Cloud Pipeline: Data Ingestion & Preprocessing
We definiëren vooraf een expliciet schema (`StructType`) voor de CSV-data. Dit is een best practice voor PySpark om schaalbaarheid te garanderen. We vullen missende temperatuurwaarden op met de mediaan, en ontbrekende neerslag (`rain_1h`, `snow_1h`) met `0.0`.

In [0]:
# Expliciet Schema Definiëren
metro_schema = StructType([
    StructField("holiday", StringType(), True),
    StructField("temp", DoubleType(), True),
    StructField("rain_1h", DoubleType(), True),
    StructField("snow_1h", DoubleType(), True),
    StructField("clouds_all", IntegerType(), True),
    StructField("weather_main", StringType(), True),
    StructField("weather_description", StringType(), True),
    StructField("date_time", StringType(), True),
    StructField("traffic_volume", IntegerType(), True)
])

def clean_cloud_data(df):
    df = df.dropDuplicates()
    df = df.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))
    
    df = df.withColumn("year", year(col("date_time"))) \
           .withColumn("month", month(col("date_time"))) \
           .withColumn("day", dayofmonth(col("date_time"))) \
           .withColumn("hour", hour(col("date_time"))) \
           .withColumn("dayofweek", dayofweek(col("date_time")))

    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(c, regexp_replace(col(c), r"\s+", " "))
        df = df.fillna({c: "Unknown"})

    median_temp = df.approxQuantile("temp", [0.5], 0.01)[0]
    median_clouds = df.approxQuantile("clouds_all", [0.5], 0.01)[0]
    df = df.fillna({"temp": median_temp, "clouds_all": median_clouds})
    
    df = df.fillna({"rain_1h": 0.0, "snow_1h": 0.0})
    df = df.dropna(subset=["traffic_volume"])

    return df

# Inladen met paden uit configuratie cel
df_train = clean_cloud_data(spark.read.schema(metro_schema).option("header", True).csv(TRAIN_DATA_PATH))
df_test = clean_cloud_data(spark.read.schema(metro_schema).option("header", True).csv(TEST_DATA_PATH))

df_train.show(5)

In [0]:
# Stel lokale MLflow tracking in
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("Cloud_Traffic_Forecasting")

feature_cols = ["temp", "rain_1h", "snow_1h", "clouds_all", "month", "hour", "dayofweek"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="skip")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

rf = RandomForestRegressor(featuresCol="features", labelCol="traffic_volume", seed=42)
cloud_pipeline = Pipeline(stages=[assembler, scaler, rf])

paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [20, 50])
             .addGrid(rf.maxDepth, [5, 10])
             .build())

evaluator = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="rmse")

cv = CrossValidator(estimator=cloud_pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3)

with mlflow.start_run(run_name="RandomForest_Tuning_Local"):
    print("Start training & tuning cloud model. Dit kan even duren...")
    
    cv_model = cv.fit(df_train)
    best_model = cv_model.bestModel
    predictions = best_model.transform(df_test)
    
    rmse = evaluator.evaluate(predictions)
    mae = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="mae").evaluate(predictions)
    
    best_rf = best_model.stages[-1]
    mlflow.log_param("best_numTrees", best_rf.getNumTrees)
    mlflow.log_param("best_maxDepth", best_rf.getOrDefault('maxDepth'))
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAE", mae)
    
    print(f"Training Cloud Model voltooid, RMSE: {rmse:.2f}, MAE: {mae:.2f}")

In [0]:
mlflow.set_experiment("YOLOv8_Traffic_Detection")

with mlflow.start_run(run_name="yolov8n_edge_model_local"):
    
    actual_epochs = 50
    actual_batch = 16
    actual_optimizer = "AdamW"
    
    mlflow.log_param("model", "yolov8n")
    mlflow.log_param("epochs", actual_epochs)
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("batch", actual_batch)
    mlflow.log_param("optimizer", actual_optimizer)

    model = YOLO("yolov8n.pt")

    print("Start YOLO training lokaal...")
    start_time = time.time()

    # Zorg dat je 'data.yaml' ook relatieve lokale paden heeft!
    yolo_results = model.train(
        data="data.yaml",
        epochs=actual_epochs,
        imgsz=640,
        batch=actual_batch,
        optimizer=actual_optimizer,
        workers=0, # Voorkomt crashes in VS code op Windows/Mac!
        project="traffic_yolo",
        name="yolov8n_edge"
    )

    training_time = time.time() - start_time
    metrics = model.val()

    mlflow.log_metric("training_time_seconds", training_time)
    mlflow.log_metric("mAP50", float(metrics.box.map50))

    # --- TFLITE EXPORT VOOR EDGE DEPLOYMENT ---
    print("Start export naar TensorFlow Lite (INT8)...")
    best_model = YOLO("traffic_yolo/yolov8n_edge/weights/best.pt")
    
    # Export met INT8 Quantization voor <10MB requirement
    export_path = best_model.export(format="tflite", int8=True)
    
    if os.path.exists(export_path):
        tflite_size_mb = os.path.getsize(export_path) / (1024 * 1024)
        mlflow.log_metric("tflite_model_size_mb", tflite_size_mb)
        mlflow.log_artifact(export_path, artifact_path="yolo_tflite_edge_model")
        print(f"TFLite export succesvol. Modelgrootte: {tflite_size_mb:.2f} MB")

Maak in VS Code een nieuwe map src/api/ en maak daarin een bestand genaamd main.py.

In [0]:
from fastapi import FastAPI
from pydantic import BaseModel
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
import pandas as pd

# 1. Definieer de API
app = FastAPI(
    title="Nova Stad Traffic Voorspeller",
    description="API voor het voorspellen van verkeersdrukte o.b.v. weerdata."
)

# 2. Definieer hoe de input data eruit moet zien
class TrafficRequest(BaseModel):
    temp: float
    rain_1h: float
    snow_1h: float
    clouds_all: int
    month: int
    hour: int
    dayofweek: int

# We starten Spark en laden het model zodra de API opstart
spark = SparkSession.builder.master("local[1]").appName("TrafficAPI").getOrCreate()

# LET OP: In de praktijk laad je hier je opgeslagen model in:
# model = PipelineModel.load("../../models/cloud_hybrid/random_forest_model")

@app.get("/")
def home():
    return {"message": "De Nova Stad Traffic API is online! Ga naar /docs voor de interface."}

@app.post("/predict")
def predict_traffic(data: TrafficRequest):
    # Converteer de inkomende JSON request naar een Pandas DataFrame en dan naar Spark
    pdf = pd.DataFrame([data.dict()])
    sdf = spark.createDataFrame(pdf)
    
    # In productie doe je hier: prediction = model.transform(sdf).collect()[0]["prediction"]
    # Voor nu simuleren we een antwoord:
    gesimuleerde_voorspelling = 3500 
    
    return {
        "status": "success",
        "verwachte_verkeersvolume": gesimuleerde_voorspelling,
        "gebruikte_features": data.dict()
    }

Maak in de allerhoogste map van je project (naast je README.md) een bestand genaamd Dockerfile (geen extensie!).

In [0]:
# Gebruik een lichte Python basis-image
FROM python:3.9-slim

# Omdat we PySpark gebruiken, hebben we Java nodig in de container
RUN apt-get update && \
    apt-get install -y default-jre && \
    apt-get clean

# Stel de werkmap in
WORKDIR /app

# Kopieer de benodigdheden en installeer deze
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Kopieer de source code en eventuele modellen
COPY src/ /app/src/

# Open poort 8000 voor de API
EXPOSE 8000

# Start de FastAPI server wanneer de container start
CMD ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]

We gaan GitHub zo instellen dat hij bij elke push automatisch checkt of je code netjes is (volgens PEP8) en of het werkt.
Maak de mappen .github/workflows/ aan en maak daarin het bestand ci_cd.yml.

In [0]:
name: Nova Stad MLOps Pipeline (CI/CD)

on:
  push:
    branches: [ "main" ]

jobs:
  build_and_test:
    runs-on: ubuntu-latest

    steps:
    - name: Code ophalen
      uses: actions/checkout@v3

    - name: Python instellen
      uses: actions/setup-python@v4
      with:
        python-version: "3.9"

    - name: Dependencies installeren
      run: |
        python -m pip install --upgrade pip
        pip install flake8 pytest
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Linting met Flake8 (Check PEP8)
      run: |
        # Stop de build als er ernstige syntax fouten zijn
        flake8 src/ --count --select=E9,F63,F7,F82 --show-source --statistics

"Monitoren en Drift detectie". Hiervoor gaan we terug naar je Jupyter Notebook. Voeg aan het einde van je notebook deze 2 cellen toe. Installeer eerst even het package via je terminal: pip install evidently.

## 6. Monitoren & Data Drift Detectie (MLOps)
Om te voldoen aan de monitoring-eisen van een productie-systeem, moeten we detecteren of de data over tijd verandert (Data Drift). Als het bijvoorbeeld in de test-dataset plotseling veel meer regent dan in de train-dataset, zal ons model slechter presteren.

We gebruiken **Evidently AI** om een geautomatiseerd drift-dashboard te genereren. Indien er significante drift wordt gedetecteerd in kritieke variabelen (zoals `temp` of `rain_1h`), dient dit als trigger om de Continuous Training (CT) pijplijn te activeren.

In [0]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

print("Start Data Drift Analyse...")

# Evidently AI werkt optimaal met Pandas.
# Om geheugen te besparen converteren we een representatieve sample (10.000 rijen) van Spark naar Pandas.
features_to_monitor = ["temp", "rain_1h", "clouds_all", "traffic_volume"]

train_sample_pd = df_train.select(features_to_monitor).limit(10000).toPandas()
test_sample_pd = df_test.select(features_to_monitor).limit(10000).toPandas()

# Genereer het Data Drift Report
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(reference_data=train_sample_pd, current_data=test_sample_pd)

# Optie 1: Sla het dashboard op als een interactieve HTML pagina (zeer professioneel voor MLOps logging!)
drift_report.save_html("data_drift_dashboard.html")
print("Dashboard succesvol opgeslagen als 'data_drift_dashboard.html'")

# Optie 2: Toon het direct in dit notebook (Werkt perfect in VS Code!)
drift_report.show(mode='inline')